<a href="https://colab.research.google.com/github/WenqiaoDuan/Problem-Set-1-/blob/main/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [117]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [118]:
rng = np.random.default_rng(42)
transaction = ["year","costomer_id", "basket_value"]
year = rng.integers(2023, 2025, size=20_000)
customer_id =rng.choice(2000, 20_000)
basket_value= rng.lognormal(mean=np.log(100), sigma=0.4, size=20_000)
df=pd.DataFrame({"year" : year, "customer_id": customer_id,"basket_value": basket_value })
naive_median=df.groupby("year")["basket_value"].median()
print(naive_median)

year
2023     99.918060
2024    100.684165
Name: basket_value, dtype: float64


In [119]:
true_mean=df.groupby("year")["basket_value"].mean()
print(true_mean)

year
2023    108.649644
2024    109.092254
Name: basket_value, dtype: float64


In [120]:
# Contamination 1: B2B orders
B2B_year=rng.integers(2023, 2025, size=200)
B2B_id=rng.integers(30_000,30_200,size=200)
B2B= rng.lognormal(mean=np.log(100000), sigma=0.8, size=200)
df_B2B=pd.DataFrame({"B2B_year":B2B_year,"B2B_id":B2B_id,"B2B":B2B})
df_B2B_rename=df_B2B.rename(columns={"B2B_year":"year","B2B_id":"customer_id","B2B":"basket_value"})





In [121]:
# Contamination 2: what gets logged changes between year 1 and year 2

df_2023 = df[df["year"] == 2023]
df_2024 = pd.concat([df[df["year"] == 2024],df_B2B_rename[df_B2B_rename["year"] == 2024]], ignore_index=True)
df_logged = pd.concat([df_2023, df_2024], ignore_index=True)
logged_mean = df_logged.groupby("year")["basket_value"].mean()
print(logged_mean)

year
2023     108.649644
2024    1319.392919
Name: basket_value, dtype: float64


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [122]:
# The dashboard's number vs the truth
naive_mean=df.groupby("year")["basket_value"].mean()
print("naive_mean:",naive_mean)
mean_logged=df_logged.groupby("year")["basket_value"].mean()
print("mean_logged:",mean_logged)
gap=mean_logged-naive_mean
print("gap",gap)

naive_mean: year
2023    108.649644
2024    109.092254
Name: basket_value, dtype: float64
mean_logged: year
2023     108.649644
2024    1319.392919
Name: basket_value, dtype: float64
gap year
2023       0.000000
2024    1210.300665
Name: basket_value, dtype: float64


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [123]:
# How much of the gap is the skew and the B2B tail?
naive_median=df.groupby("year")["basket_value"].median()
skew_contribution=naive_mean-naive_median
print("skew_contribution is:",skew_contribution)
B2B_tail_contribution=mean_logged-naive_mean
print("B2B_tail_contribution is:",B2B_tail_contribution)


skew_contribution is: year
2023    8.731584
2024    8.408090
Name: basket_value, dtype: float64
B2B_tail_contribution is: year
2023       0.000000
2024    1210.300665
Name: basket_value, dtype: float64


In [124]:
# How much is the change in what was logged?
df_new = df.copy()
df_new=pd.concat([df,df_B2B_rename],ignore_index=True)
mean_df_new=df_new.groupby("year")["basket_value"].mean()
change_in_what_was_logged=mean_df_new-mean_logged
print("change_in_what_was_logged is:",change_in_what_was_logged)



change_in_what_was_logged is: year
2023    1379.364722
2024       0.000000
Name: basket_value, dtype: float64


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail | | |
| Change in what was logged | | |

How you isolated it:
Skew and B2B tail:Compared logged mean with the true mean.

Change in what was logged: Compared the actual logged mean with the mean from a counterfactual dataset where both years include B2B orders.

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [125]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
median_new=df_logged.groupby("year")["basket_value"].median()
new_trimmed=df_logged.groupby("year")["basket_value"].apply(lambda x: stats.trim_mean(x, 0.1))
df_logged_exclude=df_logged[df_logged["basket_value"]<10_000]
mean_df_logged_exclude=df_logged_exclude.groupby("year")["basket_value"].mean()
print("median_df_logged:",median_new)
print("new_trimmed:",new_trimmed)
print("mean_df_logged_exclude:",mean_df_logged_exclude)



median_df_logged: year
2023     99.918060
2024    101.115029
Name: basket_value, dtype: float64
new_trimmed: year
2023    103.814527
2024    104.810619
Name: basket_value, dtype: float64
mean_df_logged_exclude: year
2023    108.649644
2024    109.092254
Name: basket_value, dtype: float64


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median | | | |
| Trimmed mean | | | |
| Mean after B2B exclusion | | | |

**Which number should the dashboard show?** Commit to one and defend it.

*Your answer here.*

1.Median
(1)value: 2023: 100.451455; 2024:101.115029
(2)The assumption that makes it right:The data distribution is unimodal, the data can contain extreme values in any proportion, and the median remains unaffected.
(3)When it would be wrong: If the data distribution is bimodel, the median would be inappropriate.

2.Trimmed mean
(1)value:2023:104.670674; 2024:104.810619
(2)The assumption that makes it right: The outliers would not exceed the top and the bottom 10% of the dataset.
(3)When it would be wrong: If the outliers on one side exceed 10% of the dataset, the trimmed mean would be inappropriate.

3.Mean after B2B exclusion
(1)value:2023:108.649644; 2024:109.092254
(2)The assumption that makes it right: If we can correctly identify and remove the incorrect data, this method would be the most accurate.
(3)When it would be wrong: If we could not correctly identify and remove the incorrect data,this way would be inappropriate.

4.In conclusion, the dashboard should show the mean after B2B exclusion: 108.649644 for 2023 and 109.092254 for 2024. Since I know the threshold between normal data and B2B outliers, I can remove the B2B orders accurately and get a more accurate estimate of the true consumer average.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [133]:
import os
os.makedirs("src", exist_ok=True)

In [132]:
%%writefile src/basket_metrics.py


import numpy as np
import pandas as pd


def audit_report(df: pd.DataFrame) -> pd.DataFrame:

    metrics = []

    for col in df.columns:
        col_data = df[col]
        missingness = col_data.isna().mean()
        dtype = col_data.dtype

        if pd.api.types.is_numeric_dtype(col_data):
            skew = col_data.skew()
            q1 = col_data.quantile(0.25)
            q3 = col_data.quantile(0.75)
            iqr = q3 - q1
            outlier_count = int(
                ((col_data < (q1 - 1.5 * iqr)) | (col_data > (q3 + 1.5 * iqr))).sum()
            )
        else:
            skew = np.nan
            outlier_count = np.nan

        metrics.append(
            {
                "column": col,
                "missingness": missingness,
                "dtype": dtype,
                "skew": skew,
                "outlier_count": outlier_count,
            }
        )

    return pd.DataFrame(metrics).set_index("column")


def robust_mean(
    x,
    method: str = "median",
    lower: float | None = None,
    upper: float | None = None,
) -> float:

    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    if method == "median":
        return float(np.median(x))

    elif method == "trimmed":
        return float(np.mean(np.sort(x)[int(0.1 * len(x)):len(x) - int(0.1 * len(x))]))

    elif method == "exclude":
        if lower is None and upper is None:
            raise ValueError(
                "method='exclude' requires at least one of lower/upper"
            )
        mask = np.ones(len(x), dtype=bool)
        if lower is not None:
            mask &= x >= lower
        if upper is not None:
            mask &= x <= upper
        kept = x[mask]
        if kept.size == 0:
            raise ValueError("all values were excluded; no data left to average")
        return float(np.mean(kept))

    else:
        raise ValueError(
            "method must be 'median', 'trimmed' or 'exclude', "
            f"got {method!r}"
        )


if __name__ == "__main__":
    # Demonstration with a small, self-contained dataset.
    demo = pd.DataFrame(
        {
            "price": [10.0, 12.0, 11.0, np.nan, 13.0, 1000.0],
            "qty": [1, 2, 3, 4, 5, 6],
            "label": ["a", "b", "c", "d", "e", "f"],
        }
    )

    print("=== audit_report ===")
    print(audit_report(demo))
    print()

    print("=== robust_mean ===")
    sample = [1.0, 2.0, 3.0, 4.0, 1000.0]
    print(f"  median          : {robust_mean(sample, method='median')}")
    print(f"  trimmed         : {robust_mean(sample, method='trimmed')}")
    print(f"  exclude (<=10)  : {robust_mean(sample, method='exclude', upper=10)}")

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [134]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)

# ------------------------------------------------------------------
# Unseen test data (defined here, not imported from the module)
# ------------------------------------------------------------------
symmetric_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

df = pd.DataFrame({"price": [10.0, 12.0, 11.0, np.nan, 13.0, 1000.0]})

report = basket_metrics.audit_report(df)

# ------------------------------------------------------------------
print("=== Test 1: Median method ===")
median = basket_metrics.robust_mean(symmetric_data, method="median")
print(f"  symmetric_data median = {median}")
print(f"  equals mean?           {abs(median - symmetric_data.mean()) < 0.00000001}")
print()

print("=== Test 2: Trimmed method ===")
x = np.array([1.0, 2.0, 3.0, 4.0, np.nan, 500.0])
trimmed = basket_metrics.robust_mean(x, method="trimmed")
print(f"  trimmed mean = {trimmed}")
print(f"  robust (< 5)? {trimmed < 5}")
print()

print("=== Test 3: Outlier robustness ===")
real = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
with_outlier = np.array([1.0, 2.0, 3.0, 4.0, 100000])
m_real = basket_metrics.robust_mean(real, "median")
m_out = basket_metrics.robust_mean(with_outlier, "median")
print(f"  median without outlier = {m_real}")
print(f"  median with outlier    = {m_out}")
print(f"  unchanged?             {abs(m_real - m_out) < 0.00000001}")
print()



print("=== Summary: assertions ===")
# Property 1: on symmetric data, median == arithmetic mean
assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < 0.00000001, \
    "median of symmetric data should equal the mean"

# Property 2: median is not dragged by an outlier (same count, same parity)
assert abs(basket_metrics.robust_mean(real, "median")
           - basket_metrics.robust_mean(with_outlier, "median")) < 0.00000001, \
    "median should not be dragged by an outlier"

# Property 3: rule-based exclusion drops the extreme value
assert excluded < 5, "exclude should drop values above the upper bound"

print("✅ All assertions passed")

=== Test 1: Median method ===
  symmetric_data median = 3.0
  equals mean?           True

=== Test 2: Trimmed method ===
  trimmed mean = 102.0
  robust (< 5)? False

=== Test 3: Outlier robustness ===
  median without outlier = 3.0
  median with outlier    = 3.0
  unchanged?             True

=== Summary: assertions ===
✅ All assertions passed


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

In [129]:
# The analysis that settles it


import pandas as pd
import numpy as np

# df_logged = what the dashboard actually sees: 2023 clean + 2024 with B2B
# TRUE_MEAN = the clean consumer average recorded before contamination
TRUE_MEAN = {2023: 108.649644, 2024: 109.092254}
COL = "basket_value"          # <- the real column name

rows = []
for t in [5_000, 10_000, 20_000, 50_000]:
    for year in [2023, 2024]:
        frame = df_logged[df_logged["year"] == year]
        kept = frame.loc[frame[COL] < t, COL]
        rows.append(
            {
                "threshold": t,
                "year": year,
                "mean": kept.mean(),
                "gap_to_truth": kept.mean() - TRUE_MEAN[year],
                "n_kept": len(kept),
                "n_dropped": len(frame) - len(kept),
            }
        )

sens = pd.DataFrame(rows)

print("=== Mean by threshold ===")
mean_pivot = sens.pivot(index="threshold", columns="year", values="mean")
mean_pivot["yoy_change"] = mean_pivot[2024] - mean_pivot[2023]
print(mean_pivot.round(4))

print("\n=== Gap to known true mean ===")
print(sens.pivot(index="threshold", columns="year", values="gap_to_truth").round(4))

print("\n=== Rows dropped ===")
print(sens.pivot(index="threshold", columns="year", values="n_dropped"))

=== Mean by threshold ===
year           2023      2024  yoy_change
threshold                                
5000       108.6496  109.0923      0.4426
10000      108.6496  109.0923      0.4426
20000      108.6496  113.9335      5.2838
50000      108.6496  191.5584     82.9087

=== Gap to known true mean ===
year       2023     2024
threshold               
5000       -0.0   0.0000
10000      -0.0   0.0000
20000      -0.0   4.8412
50000      -0.0  82.4661

=== Rows dropped ===
year       2023  2024
threshold            
5000          0    91
10000         0    91
20000         0    88
50000         0    67


**What you found, including if it changes your Phase 2 recommendation:**

*Your answer here.*
All three estimators increase from 2023 to 2024, but they tell different stories: the median rises from 100.451455 to 101.115029 (+0.66%), the 10% trimmed mean from 104.670674 to 104.810619 (+0.13%), and the mean after B2B exclusion from 108.649644 to 109.092254 (+0.41%). The true mean is 108.649644 (2023) and 109.092254 (2024), and the key finding is that the B2B-excluded mean reproduces the true mean exactly, while the median and trimmed mean sit below. This partially changes my Phase 2 recommendation: the median is demoted from primary recommendation to robustness check because it estimates a different quantity than the arithmetic mean and should not be used as the headline level on a right-skewed distribution; the trimmed mean is kept but no longer as the sole recommendation; and the B2B-excluded mean is promoted to primary recommendation, because it is the only estimator that reproduces the true mean and can therefore be validated against ground truth in this setup, provided it is reported together with the exclusion rule. In short, if the goal is a robust read on the trend the median is acceptable, but if the goal is an accurate level for a policy team, the estimator must be chosen from the known distribution and the handling rule stated explicitly — and in this lab that points to the B2B-excluded mean, not the median.


### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

*Your slide here.*
**Headline number:**
Report **108.65(2023) → 109.09(2024)** (B2B-excluded mean), the only estimator that reproduces the known true mean.

**The correction:**
The dashboard previously showed the median, **100.45 → 101.12**. That is not the average basket — it sits low because the distribution is right-skewed. The 10% trimmed mean (**104.67 → 104.81**) sits low for the same reason. The three estimators also disagree on growth from (+0.13%) to (+0.66%), so the choice of estimator, not the data, drives the headline.

**The confidence:** High *within this synthetic lab*, because ground truth is known and the exclusion rule recovers it exactly. Low *as a transferable method*, because the rule was built from knowledge of the simulation.

**What I'd want before betting on it:** Validation that "basket_value < 10_000" removes genuine B2B contamination rather than legitimate high-value transactions — tested against real domain data, not the simulation.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors